# Customer Satisfaction Survey Participation Analysis

## Notebook 00 — Create Synthetic Dataset

**Author:** Pei-Pei Lei

**Last Updated:** 2026-06-28

---

## Project Objective

Real-world survey projects often contain confidential customer information that cannot be shared publicly. To create a fully reproducible portfolio while maintaining data privacy, this notebook generates a synthetic individual-level survey participation dataset that preserves the key analytical characteristics of the original project without exposing any real customer records.

The synthetic dataset is designed to support the complete analytical workflow used throughout this repository, including exploratory analysis, statistical testing, and predictive modeling.

Specifically, this notebook will:

1.  Generate synthetic customer-level records that mimic realistic survey participation behavior.
2. Preserve key response patterns across survey year, customer segment, and geographic region.
3. Validate that the synthetic dataset accurately reproduces the intended response distributions and response rates.
4. Export a reusable analysis dataset for downstream notebooks.

## Primary Deliverable: 
A validated public synthetic dataset for portfolio use.

## Methodological Design

This notebook follows a design-based synthetic data generation approach. The goal is not to reproduce the original project records, but to create a public analysis dataset that preserves the key analytical patterns needed for downstream analysis.

### Design Decision 1 — Preserve Analytical Patterns

The synthetic dataset was designed to preserve the response patterns most relevant to this project:

- Survey participation by year
- Response rates by participation segment
- Geographic response patterns
- Project-level unit of analysis

Each row represents a project-level survey participation record. Customers may appear in multiple records if they are associated with multiple projects.

### Design Decision 2 — Use a Proportional Synthetic Sample Size

A total of **1,200 synthetic project-level records** were generated.

This sample size was selected to provide enough granularity for the observed synthetic response rates to closely match the intended design response rates. Increasing the synthetic sample size helps reduce rounding error when translating response-rate targets into individual project-level records.

The objective was not to replicate the original project size. Instead, the synthetic sample size was chosen to preserve analytical fidelity while keeping the dataset lightweight, reproducible, and appropriate for a public GitHub portfolio.

### Design Decision 3 — Validate Against Design Targets

After generating the synthetic records, the dataset was validated by comparing the observed synthetic response rates against the predefined design response rates.

This validation step confirms that the synthetic dataset preserves the intended participation patterns while ensuring that no real customer-level or project-level records are included.

```mermaid
flowchart LR

A["Define Design<br/>Targets"]
B["Choose<br/>Sample Size"]
C["Generate<br/>Records"]
D["Validate<br/>Results"]
E["Export<br/>Dataset"]

A --> B --> C --> D --> E
```

## Design Principles

This synthetic dataset was designed to balance two objectives:

- Protect confidential customer information.
- Preserve the analytical properties required for meaningful statistical analysis.

Although individual records are synthetic, the dataset maintains realistic response behavior, allowing the complete analytical workflow to be reproduced without exposing proprietary data.


In [2]:
import pandas as pd
import numpy as np
import os

In [15]:
from pathlib import Path

# ---------------------------------------------------------------------
# Project Paths
# ---------------------------------------------------------------------

PROJECT_ROOT = Path.cwd().parent

DATA_DIR = PROJECT_ROOT / "data"
DATA_RAW = DATA_DIR / "raw"
DATA_PROCESSED = DATA_DIR / "processed"

DATA_OUTPUTS = PROJECT_ROOT / "outputs"

# Create output directories if they do not exist
DATA_PROCESSED.mkdir(parents=True, exist_ok=True)
DATA_OUTPUTS.mkdir(parents=True, exist_ok=True)

In [4]:
np.random.seed(42)

## Define Synthetic Data Design

The synthetic dataset includes two survey years:

- FY25
- FY26

Each record represents one synthetic customer survey case.

Variables:

- `customer_id`
- `survey_year`
- `customer_segment`
- `in_out_state`
- `responded`

In [5]:
## create data frame design
design = pd.DataFrame({
    "survey_year": [
        "FY25", "FY25", "FY25", "FY25", "FY25", "FY25",
        "FY26", "FY26", "FY26", "FY26", "FY26", "FY26"
    ],
    "customer_segment": [
        "Silent Customers", "Silent Customers",
        "Prior Responders", "Prior Responders",
        "New Customers", "New Customers",
        "Silent Customers", "Silent Customers",
        "Prior Responders", "Prior Responders",
        "New Customers", "New Customers"
    ],
    "in_out_state": [
        "In-State", "Out-of-State",
        "In-State", "Out-of-State",
        "In-State", "Out-of-State",
        "In-State", "Out-of-State",
        "In-State", "Out-of-State",
        "In-State", "Out-of-State"
    ],
    "n_synthetic": [
        120, 30,
        210, 40,
        180, 20,
        150, 30,
        200, 50,
        140, 30
    ],
    "response_rate": [
        0.25, 0.44,
        0.66, 0.86,
        0.35, 0.52,
        0.22, 0.09,
        0.68, 0.81,
        0.30, 0.73
    ]
})

design

,survey_year,customer_segment,in_out_state,n_synthetic,response_rate
0,FY25,Silent Customers,In-State,120,0.25
1,FY25,Silent Customers,Out-of-State,30,0.44
2,FY25,Prior Responders,In-State,210,0.66
3,FY25,Prior Responders,Out-of-State,40,0.86
4,FY25,New Customers,In-State,180,0.35
5,FY25,New Customers,Out-of-State,20,0.52
6,FY26,Silent Customers,In-State,150,0.22
7,FY26,Silent Customers,Out-of-State,30,0.09
8,FY26,Prior Responders,In-State,200,0.68
9,FY26,Prior Responders,Out-of-State,50,0.81


## Generate Individual-Level Synthetic Records

In [6]:
records = []

customer_counter = 1

for _, row in design.iterrows():

    n = int(row["n_synthetic"])
    response_rate = row["response_rate"]

    # Exact number of respondents
    n_response = round(n * response_rate)
    n_nonresponse = n - n_response

    # Create response vector
    responses = np.array(
        [1] * n_response +
        [0] * n_nonresponse
    )

    # Randomize order
    np.random.shuffle(responses)

    # Create synthetic records
    for responded in responses:

        records.append({
            "customer_id": f"C{customer_counter:06d}",
            "survey_year": row["survey_year"],
            "customer_segment": row["customer_segment"],
            "in_out_state": row["in_out_state"],
            "responded": responded
        })

        customer_counter += 1

synthetic = pd.DataFrame(records)

synthetic.head()

,customer_id,survey_year,customer_segment,in_out_state,responded
0,C000001,FY25,Silent Customers,In-State,0
1,C000002,FY25,Silent Customers,In-State,0
2,C000003,FY25,Silent Customers,In-State,1
3,C000004,FY25,Silent Customers,In-State,0
4,C000005,FY25,Silent Customers,In-State,1


In [7]:
synthetic.shape

(1200, 5)

## Validate Synthetic Response Patterns

In [8]:
validation = (
    synthetic
    .groupby(
        ["survey_year",
         "customer_segment",
         "in_out_state"]
    )
    .agg(
        n=("customer_id", "count"),
        n_responded=("responded", "sum"),
        response_rate=("responded", "mean")
    )
    .reset_index()
)

validation["response_rate"] = (
    validation["response_rate"] * 100
).round(1)

validation

,survey_year,customer_segment,in_out_state,n,n_responded,response_rate
0,FY25,New Customers,In-State,180,63,35.0
1,FY25,New Customers,Out-of-State,20,10,50.0
2,FY25,Prior Responders,In-State,210,139,66.2
3,FY25,Prior Responders,Out-of-State,40,34,85.0
4,FY25,Silent Customers,In-State,120,30,25.0
5,FY25,Silent Customers,Out-of-State,30,13,43.3
6,FY26,New Customers,In-State,140,42,30.0
7,FY26,New Customers,Out-of-State,30,22,73.3
8,FY26,Prior Responders,In-State,200,136,68.0
9,FY26,Prior Responders,Out-of-State,50,40,80.0


## Validation Report
Compare synthetic dataset to design data attributes

In [9]:
validation_report = (
    design.merge(
        validation,
        on=[
            "survey_year",
            "customer_segment",
            "in_out_state"
        ]
    )
)

validation_report["Designed RR (%)"] = (
    validation_report["response_rate_x"] * 100
).round(1)

validation_report["Synthetic RR (%)"] = (
    validation_report["response_rate_y"]
).round(1)

validation_report["Difference"] = (
    validation_report["Synthetic RR (%)"]
    - validation_report["Designed RR (%)"]
)

validation_report = validation_report[
    [
        "survey_year",
        "customer_segment",
        "in_out_state",
        "Designed RR (%)",
        "Synthetic RR (%)",
        "Difference"
    ]
]

validation_report

,survey_year,customer_segment,in_out_state,Designed RR (%),Synthetic RR (%),Difference
0,FY25,Silent Customers,In-State,25.0,25.0,0.0
1,FY25,Silent Customers,Out-of-State,44.0,43.3,-0.7
2,FY25,Prior Responders,In-State,66.0,66.2,0.2
3,FY25,Prior Responders,Out-of-State,86.0,85.0,-1.0
4,FY25,New Customers,In-State,35.0,35.0,0.0
5,FY25,New Customers,Out-of-State,52.0,50.0,-2.0
6,FY26,Silent Customers,In-State,22.0,22.0,0.0
7,FY26,Silent Customers,Out-of-State,9.0,10.0,1.0
8,FY26,Prior Responders,In-State,68.0,68.0,0.0
9,FY26,Prior Responders,Out-of-State,81.0,80.0,-1.0


## Export Synthetic Dataset

The synthetic dataset is exported to the `data/raw` folder for use in later notebooks.

In [10]:
os.getcwd() 

'/Users/peipei/Documents/GitHub/customer-satisfaction-survey-participation-analysis/notebooks'

In [16]:
# Export to DATA_RAW (defined on top of this notebook) to make sure data/raw is the destination folder

synthetic.to_csv(
    DATA_RAW / "survey_participation_data.csv",
    index=False
)

validation.to_csv(
    DATA_RAW / "validation_summary.csv",
    index=False
)

print(f"Synthetic dataset exported to:\n{DATA_RAW}")

Synthetic dataset exported to:
/Users/peipei/Documents/GitHub/customer-satisfaction-survey-participation-analysis/data/raw
